In [16]:
%matplotlib qt
import sys
from pathlib import Path

src_path = Path.cwd().parent
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import time
import torch
import json
import gymnasium as gym
from envs.env import MiniGridEnvWrapper

from models.ppo_lstm import PPOLSTMAgent
from models.ride import RIDEAgent
from training.train import train_ppo_lstm, train_ride
from training.evaluate import evaluate_agent

In [ ]:
env_ids = [
    "MiniGrid-DoorKey-9x9-v0",
    "MiniGrid-SimpleCrossingS9N1-v0",
    "MiniGrid-LavaCrossingS9N2-v0",
    "MiniGrid-MultiRoom-N2-S4-v0",
    "MiniGrid-MultiRoom-N4-S5-v0",
    "MiniGrid-KeyCorridorS3R3-v0"
]
env = MiniGridEnvWrapper(env_ids[3], render_mode='rgb_array')

obs, info = env.reset()
print(f"Initial observation shape: {obs.shape}")
print(f"Action space: {env.action_space}")

num_steps = 100
for step in range(num_steps):
    action = env.action_space.sample()
    
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Step {step + 1}: Action={action}, Reward={reward}, Done={terminated or truncated}")
    
    env.display_interactive()
    
    if terminated or truncated:
        print(f"Episode finished after {step + 1} steps!")
        obs, info = env.reset()
        print("Environment reset for next episode")
        env.display_interactive()
    time.sleep(0.01)
env.close()

Initial observation shape: (7, 7, 3)
Action space: Discrete(7)
Step 1: Action=4, Reward=0, Done=False
Step 2: Action=2, Reward=0, Done=False
Step 3: Action=1, Reward=0, Done=False
Step 4: Action=0, Reward=0, Done=False
Step 5: Action=1, Reward=0, Done=False
Step 6: Action=0, Reward=0, Done=False
Step 7: Action=3, Reward=0, Done=False
Step 8: Action=6, Reward=0, Done=False
Step 9: Action=5, Reward=0, Done=False
Step 10: Action=0, Reward=0, Done=False
Step 11: Action=6, Reward=0, Done=False
Step 12: Action=2, Reward=0, Done=False
Step 13: Action=6, Reward=0, Done=False
Step 14: Action=5, Reward=0, Done=False
Step 15: Action=0, Reward=0, Done=False
Step 16: Action=0, Reward=0, Done=False
Step 17: Action=5, Reward=0, Done=False
Step 18: Action=0, Reward=0, Done=False
Step 19: Action=6, Reward=0, Done=False
Step 20: Action=4, Reward=0, Done=False
Step 21: Action=3, Reward=0, Done=False
Step 22: Action=1, Reward=0, Done=False
Step 23: Action=3, Reward=0, Done=False
Step 24: Action=5, Reward=

In [ ]:
agent = train_ppo_lstm(
        env=env,
        experiment_name="ppo_lstm_simple_crossing_baseline",
        num_iterations=300,
        print_interval=100,  # Print every 10 iterations for cleaner output
        steps_per_iteration=2048,
        save_interval=10,
        device="cpu",  # or "cuda" or "mps"
        lr=3e-4,
        gamma=0.99,
        ppo_epochs=4,
        ppo_minibatch_size=4,
        hidden_size=256,
    )

In [ ]:
agent_class = PPOLSTMAgent
env = MiniGridEnvWrapper(env_ids[1], render_mode='rgb_array')
config_path = "../runs/ppo_lstm_simple_crossing_baseline_20251112_230418/config.json"
checkpoint_path = "../checkpoints/ppo_lstm_simple_crossing_baseline_20251112_230418/final_model.pt"

results = evaluate_agent(agent_class, env, checkpoint_path, config_path)

In [39]:
# Train RIDE agent
ride_agent = train_ride(
    env = env,
    experiment_name="ride_multiroom_n2",
    num_iterations=60,
    print_interval=1,
    steps_per_iteration=2048,
    save_interval=50,
    device="cpu",  # or "cuda" or "mps"
    lr=3e-4,
    gamma=0.99,
    ppo_epochs=4,
    ppo_minibatch_size=4,
    hidden_size=256,
    intrinsic_reward_coef=0.1,  # RIDE coefficient - controls weight of intrinsic rewards
)


Logging to: ../runs/ride_multiroom_n2_20251114_215135

Starting RIDE training: ride_multiroom_n2_20251114_215135
Intrinsic reward coefficient: 0.1

[   1/60] Frames:   2,048 | Reward:    0.01 | Intrinsic:  0.462 | Length:   39.6 | Loss: -0.0191/0.0012
[   2/60] Frames:   4,096 | Reward:    0.01 | Intrinsic:  0.535 | Length:   39.7 | Loss: -0.0149/0.0008
[   3/60] Frames:   6,144 | Reward:    0.03 | Intrinsic:  0.576 | Length:   38.9 | Loss: -0.1389/0.0054
[   4/60] Frames:   8,192 | Reward:    0.01 | Intrinsic:  0.593 | Length:   39.8 | Loss: -0.0093/0.0009
[   5/60] Frames:  10,240 | Reward:    0.01 | Intrinsic:  0.589 | Length:   39.6 | Loss: -0.0213/0.0012
[   6/60] Frames:  12,288 | Reward:    0.01 | Intrinsic:  0.723 | Length:   39.7 | Loss: -0.0166/0.0012
[   7/60] Frames:  14,336 | Reward:    0.02 | Intrinsic:  0.729 | Length:   39.3 | Loss: -0.0162/0.0026
[   8/60] Frames:  16,384 | Reward:    0.05 | Intrinsic:  0.731 | Length:   38.1 | Loss: -0.0549/0.0073
[   9/60] Frames:  1

In [40]:
# Evaluate RIDE agent
ride_agent_class = RIDEAgent
env_ride_eval = MiniGridEnvWrapper(env_ids[3], render_mode='rgb_array')

# Get the most recent experiment directory
import glob
import os
ride_runs = sorted(glob.glob("../runs/ride_multiroom_n2_*"), key=os.path.getmtime, reverse=True)
if ride_runs:
    latest_run = ride_runs[0]
    config_path = os.path.join(latest_run, "config.json")
    checkpoint_path = os.path.join("../checkpoints", os.path.basename(latest_run), "final_model.pt")
    
    print(f"Using checkpoint: {checkpoint_path}")
    print(f"Using config: {config_path}")
    
    results = evaluate_agent(ride_agent_class, env_ride_eval, checkpoint_path, config_path)
else:
    print("No RIDE training runs found. Please train the agent first.")


Using checkpoint: ../checkpoints/ride_multiroom_n2_20251114_215135/final_model.pt
Using config: ../runs/ride_multiroom_n2_20251114_215135/config.json
Loading checkpoint from ../checkpoints/ride_multiroom_n2_20251114_215135/final_model.pt...
Checkpoint loaded successfully!

Running 10 evaluation episodes...
Episode  1/10 | Steps:   7 | Reward:   0.84
Episode  2/10 | Steps:   9 | Reward:   0.80
Episode  3/10 | Steps:   7 | Reward:   0.84
Episode  4/10 | Steps:   4 | Reward:   0.91
Episode  5/10 | Steps:   6 | Reward:   0.86
Episode  6/10 | Steps:   8 | Reward:   0.82
Episode  7/10 | Steps:   5 | Reward:   0.89
Episode  8/10 | Steps:   9 | Reward:   0.80
Episode  9/10 | Steps:   5 | Reward:   0.89
Episode 10/10 | Steps:   5 | Reward:   0.89

Evaluation Summary:
  Episodes:     10
  Mean Reward:  0.85 ± 0.04
  Min/Max:      0.80 / 0.91
  Mean Length:  6.50 ± 1.69
  Success Rate: 100.0%
